Reference: https://j2rooong.tistory.com/entry/Pytorch-Transformer-Architecture-구현하기

In [ ]:
import torch
import torch.nn as nn
import math

# ----------------------------------------------------------------
# 1. Input Embedding
# ----------------------------------------------------------------
class InputEmbeddings(nn.Module):
    """
    create an input embedding
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)  # matrix

    def forward(self, x):
        # matrix * constant
        # To prevent a positional embedding from shadowing the input embedding
        return self.embedding(x) * math.sqrt(self.d_model)
    
# ----------------------------------------------------------------
# 2. Positional Encoding
# ----------------------------------------------------------------
class PositionalEncoding(nn.Module):
    # This method doesnt return anything (None)
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None: 
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        
        # 2D tensor with 0s
        pe = torch.zero(seq_len, d_model)   
        # 2D tensor (seq_len, 1)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(dim=1)

        _2i = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float))

        # Giving each word an unique barcode
        # 0::2 -> from 0 to the end, with a step of 2
        pe[:, 0::2] = torch.sin(position/10000**(_2i/d_model))
        pe[:, 1::2] = torch.cos(position/10000**(_2i/d_model))

        pe = pe.unsqueeze(dim=0)  # 3D tensor (1, seq_len, d_model)
        # While sending data, a buffer keeps the data temporarily
        # With this module, the buffer is saved and loaded when the model is.
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Input x + positional encoding
        # No need to back-propagate
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        # To avoid overfitting, change some of elements into O
        return self.dropout(x)
